In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle

from sklearn.preprocessing import (
    LabelEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
import warnings


from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)

warnings.filterwarnings('ignore')

# import warnings
# warnings.filterwarnings('ignore')


In [ ]:
machine_number = 1
hours= 24

In [3]:
# machine = pd.read_csv(f"../../../data/azure_pm/machines/machine_{machine_number}.csv")

machine = pd.read_csv(f"../../../data/azure_pm/lag_features/machine_{machine_number}_lag_features.csv")

In [4]:
machine.head()

,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,error_count_6h,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error
0,2015-01-01 06:00:00,0.303137,0.604476,0.271993,0.621000,0,0,0,0,0.000000,...,0.0,0.0,0.0,0.0,6,3,0,0,0,0
1,2015-01-01 07:00:00,0.455312,0.635538,0.480756,0.645421,0,0,0,0,0.303137,...,0.0,0.0,0.0,0.0,7,3,0,0,1,1
2,2015-01-01 08:00:00,0.449470,0.540973,0.427196,0.356126,0,0,0,0,0.455312,...,0.0,0.0,0.0,0.0,8,3,0,1,2,2
3,2015-01-01 09:00:00,0.408248,0.687752,0.378658,0.254123,0,0,0,0,0.449470,...,0.0,0.0,0.0,0.0,9,3,0,1,3,3
4,2015-01-01 10:00:00,0.486041,0.518746,0.454206,0.351050,0,0,0,0,0.408248,...,0.0,0.0,0.0,0.0,10,3,0,1,4,4


In [ ]:
machine_failures = machine[machine['failure'] != '0']
machine_failures

In [ ]:
machine_failures = machine[machine['failure'] != '0']
machine_failures

In [ ]:
def rows_between_failure_and_previous_n_hours(machine, machine_failure, hours):
    """ This method outputs the rows between the failure and 24 hour before that. """

    # Get the timestamps of failures
    failure_times = machine_failure["datetime"]

    # Convert the datetime columns to datetime if they're not already
    machine["datetime"] = pd.to_datetime(machine["datetime"])
    failure_times = pd.to_datetime(failure_times)

    # Create a mask to select rows from machine_1 that fall within the 24-hour window before each failure
    mask = pd.Series(False, index=machine.index)
    for failure_time in failure_times:
        start_time = failure_time - pd.Timedelta(hours=hours)
        mask |= (machine["datetime"] >= start_time) & (machine["datetime"] <= failure_time)   #  The row for failure itself wont be existed if we usay  ... (machine["datetime"] < failure_time)
    

    # Select the rows from machine_1 that match the mask
    machine_1_prev_24h = machine.loc[mask]

    return machine_1_prev_24h

df_test = rows_between_failure_and_previous_n_hours(machine=machine, machine_failure=machine[machine['failure'] != '0'], hours=hours)
df_test

In [ ]:
def rows_n_hours_before_failure(machine, machine_failure, hours):  
           
    failure_times = machine_failure["datetime"]

    # Convert the datetime columns to datetime if they're not already
    machine["datetime"] = pd.to_datetime(machine["datetime"])
    failure_times = pd.to_datetime(failure_times)

    # Initialize an empty list to store the rows
    rows = []

    # Iterate over each failure time
    for failure_time in failure_times:
        # Calculate the time 24 hours before the failure
        target_time = failure_time - pd.Timedelta(hours=hours)
        
        # Get the row with the closest time to the target time
        closest_row = machine.loc[(machine["datetime"] - target_time).abs().idxmin()]
        
        # Append the row to the list
        rows.append(closest_row)

    # Create a new dataframe with the rows
    machine_1_prev_24h = pd.DataFrame(rows)

    return machine_1_prev_24h


df = rows_n_hours_before_failure(machine=machine, machine_failure=machine[machine['failure'] != '0'], hours=hours)
df

In [ ]:
import os

def process_all_machines_and_get_n_hours_before_failure(data_dir, hours=24):
    """
    Reads all .csv files from the given directory, applies rows_n_hours_before_failure,
    and returns a dictionary of DataFrames keyed by machine number.
    """
    results = {}
    csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    for file in csv_files:
        machine_number = file.split('_')[-1].replace('.csv', '')
        machine_path = os.path.join(data_dir, file)
        machine_df = pd.read_csv(machine_path)
        # Ensure 'failure' column is string for comparison
        machine_df['failure'] = machine_df['failure'].astype(str)
        machine_failures = machine_df[machine_df['failure'] != '0']
        if not machine_failures.empty:
            df_n_hours = rows_n_hours_before_failure(
                machine=machine_df,
                machine_failure=machine_failures,
                hours=hours
            )
            results[machine_number] = df_n_hours
        else:
            results[machine_number] = pd.DataFrame()  # No failures found
    return results


# Example usage:
data_dir = "../../../data/azure_pm/machines/"
all_machines_n_hours = process_all_machines_and_get_n_hours_before_failure(data_dir, hours)

In [ ]:
machine_lag = pd.read_csv(f"../../../data/azure_pm/lag_features/machine_2_lag_features.csv")
# display(machine_lag)
matching_indices = all_machines_n_hours["2"].index
machine_lag_matched = machine_lag.loc[matching_indices]
display(machine_lag_matched)

In [ ]:
machine_1_original = pd.read_csv(f"../../../data/azure_pm/machines/machine_2.csv")
machine_1_original[machine_1_original['failure'] != '0']


In [ ]:
machine_1_original["failure"].value_counts()

In [ ]:
# Combine all DataFrames in all_machines_n_hours into a single DataFrame
all_machines_combined = pd.concat(
    [df for df in all_machines_n_hours.values() if not df.empty],
    ignore_index=True
)

# Display the combined DataFrame
# display(all_machines_combined)

In [ ]:
all_machines_combined = all_machines_combined.sort_values('datetime')
all_machines_combined

In [ ]:
machine_number = 32

display(all_machines_n_hours[f"{machine_number}"])

machine = pd.read_csv(f"../../../data/azure_pm/machines/machine_{machine_number}.csv")
machine_failures = machine[machine['failure'] != '0']
display(machine_failures)

---